# Deep Dive into Logistic Regression: Foundations, Proofs, and Optimization

This notebook provides a comprehensive breakdown of all concepts, mathematical derivations, proofs, and loss functions discussed regarding Logistic Regression.

## 1. The Origin & Anatomy of the Sigmoid Function

### 1.1 Origin: Combining $1/x$ and $e^x$
To map the unbounded output of a linear model $z \in (-\infty, +\infty)$ to a valid probability $\hat{y} \in (0, 1)$:

1. **Odds Ratio**: Define the ratio of success probability ($p$) to failure probability ($1-p$):
   $$\text{Odds} = \frac{p}{1-p} \in (0, +\infty)$$

2. **Exponential Mapping**: Map $z \in (-\infty, +\infty)$ to positive odds space using $e^z$:
   $$\frac{p}{1-p} = e^z$$

3. **Algebraic Inversion**: Isolate $p$ directly:
   $$p = e^z(1-p) \implies p = e^z - pe^z \implies p(1 + e^z) = e^z \implies p = \frac{e^z}{1 + e^z}$$

4. **Sigmoid Form**: Divide numerator and denominator by $e^z$:
   $$p = \frac{1}{1 + e^{-z}}$$

* **Why $e^z$?**: $e^z$ continuously and smoothly maps $(-\infty, +\infty) \to (0, +\infty)$, yielding a log-linear model where unit feature changes scale odds multiplicatively. It serves as the canonical link function for Bernoulli distributions in Generalized Linear Models (GLMs).
* **Role of Hyperbola ($1/x$)**: The structural form $\frac{1}{1+X}$ stems from the reciprocal/hyperbolic function $f(X) = \frac{1}{X}$ evaluated at $X = 1 + e^{-z}$. It compresses the unbounded exponential range $(0, +\infty)$ into the bounded probability interval $(0, 1)$.

### 1.2 Key Definitions & Alternative Functions
* **Sigmoid**: "S-shaped". Refers to any smooth, monotonic curve bounded by upper and lower horizontal asymptotes at 1 and 0.
* **Bounded Range $(0,1)$**: As $z \to +\infty$, $e^{-z} \to 0 \implies \hat{y} \to 1$. As $z \to -\infty$, $e^{-z} \to +\infty \implies \hat{y} \to 0$.
* **Alternative Logistic Functions**:
  1. **Probit**: $\Phi(z)$, cumulative distribution function (CDF) of the standard normal distribution.
  2. **Hyperbolic Tangent**: $\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$, mapping $(-\infty, +\infty) \to (-1, 1)$. Note: $\sigma(z) = \frac{\tanh(z/2) + 1}{2}$.
  3. **Arctan**: $f(z) = \frac{1}{\pi} \arctan(z) + \frac{1}{2}$.
  4. **Algebraic Sigmoid**: $f(z) = \frac{z}{\sqrt{1 + z^2}}$.

## 2. Linear Component & Log Odds

### 2.1 The Linear Combination
$$z = b + w_1 x_1 + w_2 x_2 + \dots + w_n x_n = \mathbf{w}^T \mathbf{x} + b$$

### 2.2 Why $z$ is Called the Log Odds
Taking the natural logarithm of the odds ratio for $p = \sigma(z)$:
$$\text{Log Odds} = \ln\left(\frac{p}{1-p}\right)$$
Since $\frac{p}{1-p} = \frac{\frac{1}{1 + e^{-z}}}{\frac{e^{-z}}{1 + e^{-z}}} = e^z$:
$$\ln\left(\frac{p}{1-p}\right) = \ln(e^z) = z$$
Hence, $z$ represents the **log odds** of the positive outcome.

### 2.3 Evaluation vs. Optimization (Value vs. Equation)
* **Inference / Evaluation**: $z$ is computed as a concrete scalar numerical **value** for a given sample $\mathbf{x}$, which is passed into $\sigma(z)$.
* **Optimization**: $z$ is treated as an algebraic **equation** ($\mathbf{w}^T\mathbf{x} + b$) so partial derivatives $\frac{\partial L}{\partial w_i}$ can be computed analytically via the chain rule.

## 3. Mathematical Proofs

### Proof 1: Incremental Change in Linear Model
**Claim**: Given $y = b + 3x_1$, incrementing $x_1$ by $1$ increases $y$ by $3$.

* **Proof**:
  Let $y_{old} = b + 3x_1$.
  Increment $x_1$ by $1$: $x_{1, new} = x_1 + 1$.
  $$y_{new} = b + 3(x_1 + 1) = b + 3x_1 + 3$$
  $$\Delta y = y_{new} - y_{old} = (b + 3x_1 + 3) - (b + 3x_1) = 3 \quad \blacksquare$$

---

### Proof 2: Mean Log Loss Decouples Batch Size and Learning Rate
**Claim**: Using Mean Log Loss enables tuning the learning rate independently of batch size $N$.

* **Total Loss Gradient Step**:
  $$\mathbf{w}^{(t+1)} = \mathbf{w}^{(t)} - \eta \sum_{i=1}^N \nabla_\mathbf{w} \ell_i(\mathbf{w})$$
  The update magnitude scales linearly with $N$. Scaling $N$ alters the effective step size, forcing $\eta$ to be re-tuned.

* **Mean Loss Gradient Step**:
  $$\mathbf{w}^{(t+1)} = \mathbf{w}^{(t)} - \eta \left( \frac{1}{N} \sum_{i=1}^N \nabla_\mathbf{w} \ell_i(\mathbf{w}) \right)$$
  The term $\frac{1}{N} \sum_{i=1}^N \nabla_\mathbf{w} \ell_i(\mathbf{w})$ estimates the expected gradient $\mathbb{E}[\nabla_\mathbf{w} \ell(\mathbf{w})]$. Its magnitude is independent of $N$, decoupling hyperparameter tuning of $\eta$ from batch size $N$. $\blacksquare$

---

### Proof 3: Driven Loss & Asymptotic Behavior Without Regularization
**Claim**: Without regularization, loss is continuously driven towards 0 on separable data by inflating weight magnitudes.

* **Proof**:
  For a separable sample with true label $y_i = 1$, the loss is $\ell_i(\mathbf{w}) = \ln(1 + e^{-z_i})$ where $z_i = \mathbf{w}^T \mathbf{x}_i + b > 0$.
  Scaling $\mathbf{w}$ by a scalar factor $k > 1$ yields $z_i(k\mathbf{w}) = k z_i$.
  $$\lim_{k \to \infty} \ell_i(k\mathbf{w}) = \lim_{k \to \infty} \ln(1 + e^{-k z_i}) = \ln(1 + 0) = 0$$
  Gradient descent minimizing loss will drive $k \to \infty$ (forcing $\Vert{}\mathbf{w}\Vert{} \to \infty$), creating severe overconfidence, numerical instability, and overfitting unless constrained by regularization. $\blacksquare$

## 4. Loss Functions: Log Loss vs. Squared Loss & Precision Issues

### 4.1 Squared Loss vs. Log Loss
1. **Non-Convexity**: Applying Squared Loss $L = (\hat{y} - y)^2$ to $\hat{y} = \sigma(z)$ yields a non-convex surface with multiple local minima and flat regions (plateaus) where gradients vanish ($\sigma'(z) \to 0$ as $ert{}zert{} \to \infty$).
2. **Precision & Memory Demands**: Near $0$ and $1$, squared loss relies on differences like $(1 - \hat{y})^2$. When $\hat{y} = 0.999999999999999$, $(1-\hat{y})^2 = 10^{-30}$. Standard 32-bit floating-point numbers (`float32`) have ~7 decimal digits of precision and suffer from truncation ($1.0 - 10^{-15} \approx 1.0$). Preserving tiny error differences near asymptotes requires larger memory precision (e.g., float64, float128). Log Loss avoids this by operating in log-space or via Log-Sum-Exp formulations.

## 5. Regularization & Complexity Reduction

### 5.1 What is Regularization?
Techniques applied during training to prevent overfitting by penalizing overly complex models or excessively large weight values.

### 5.2 Strategies to Reduce Model Complexity
1. **$L_2$ Regularization (Ridge / Weight Decay)**:
   Adds a squared norm penalty to the objective function:
   $$L_{reg}(\mathbf{w}) = L_{LogLoss}(\mathbf{w}) + \frac{\lambda}{2} \sum_{j=1}^d w_j^2$$
   Prevents weights from growing infinitely large, solving the asymptotic collapse issue on separable features.
2. **Early Stopping**:
   Monitors loss on an independent validation set during training and halts updates as soon as validation loss begins to increase, preventing the model from fitting training noise.

## 6. Log Loss Formula, Variable Denotations, and Flowchart

### 6.1 Formula
$$\mathcal{L}(\mathbf{w}) = -\frac{1}{N} \sum_{i=1}^N \left[ y_i \ln(\hat{y}_i) + (1 - y_i) \ln(1 - \hat{y}_i) \right]$$

### 6.2 Variable Denotations
* $N$: Total number of samples in the batch/dataset.
* $y_i \in \{0, 1\}$: Ground truth binary label for sample $i$.
* $\hat{y}_i \in (0, 1)$: Predicted probability that $y_i = 1$, where $\hat{y}_i = \sigma(z_i)$.
* $\ln(\cdot)$: Natural logarithm.

### 6.3 Flowchart
```
   [ Feature Inputs: x1, x2, ... xn ]
                   │
                   ▼
     Linear Combination (z)
    z = b + w1*x1 + ... + wn*xn  <── (Log Odds)
                   │
                   ▼
        Sigmoid Function σ(z)
         y_hat = 1 / (1 + e^-z)  <── (Predicted Probability ∈ (0,1))
                   │
                   ▼
         Branch by True Label y
          /                 \
     If y = 1             If y = 0
       /                     \
  Loss = -ln(y_hat)      Loss = -ln(1 - y_hat)
       \                     /
        \                   /
         ▼                 ▼
          Combined Log Loss:
   - [ y*ln(y_hat) + (1-y)*ln(1-y_hat) ]
                   │
                   ▼
          Average Over Batch
      Mean Log Loss = (1/N) * ∑ Loss
```

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def log_loss(y_true, y_pred, eps=1e-15):
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

z = np.array([-2.0, 0.0, 2.5])
y_hat = sigmoid(z)
y_true = np.array([0, 0, 1])

print(f"Logits (z): {z}")
print(f"Probabilities (y_hat): {y_hat}")
print(f"Mean Log Loss: {log_loss(y_true, y_hat):.4f}")